In [1]:
import torch
from torch import nn
import tiktoken
from torch.utils.data import Dataset, DataLoader
import os

In [2]:
tokenizer = tiktoken.get_encoding("gpt2")

In [3]:
# Hyperparameters
config = {
    "context_length": 6,
    "dim_in": 4,
    "dim_out": 4,
    "dropout_rate": 0.1,
    "n_heads": 2,
    "batch_size": 4
}

In [4]:
class GPTDataSetV1(Dataset):
    def __init__(self, text, tokenizer, context_length, stride):
        self.input_ids = []
        self.target_ids = []

        encodings = tokenizer.encode(text)

        for i in range(0, len(encodings) - context_length, stride):
            inputs = encodings[i:i+context_length]
            targets = encodings[i+1:i+context_length+1]
            self.input_ids.append(torch.tensor(inputs))
            self.target_ids.append(torch.tensor(targets))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [5]:
def create_data_loader(text, batch_size=8, context_length=6, stride=3, shuffle=True, drop_last=True):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDataSetV1(text, tokenizer, context_length, stride)
    n_cpus = os.cpu_count()
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=n_cpus)

In [6]:
with open("the_verdict.txt") as f:
    text = f.read()

In [7]:
dataloader = create_data_loader(text, batch_size=config["batch_size"], context_length=config["context_length"])

In [8]:
len(dataloader)

481

In [9]:
data_iter = iter(dataloader)

In [10]:
first_batch = next(data_iter)

In [11]:
len(first_batch)

2

In [12]:
class InputPreprocessor(nn.Module):
    def __init__(self, n_vocab):
        super().__init__()
        self.token_embeddings = nn.Embedding(n_vocab, config["dim_in"])
        self.pos_embeddings = nn.Embedding(config["context_length"], config["dim_in"])

    def forward(self, encodings):
        num_tokens = encodings.shape[-1]
        return self.token_embeddings(encodings) + self.pos_embeddings(torch.arange(0, num_tokens))

In [14]:
class MaskedMultiheadAttention(nn.Module):
    def __init__(self, qkv_bias=False):
        super().__init__()
        
        self.context_length = config["context_length"]
        self.dim_in = config["dim_in"]
        self.dim_out = config["dim_out"]
        self.n_heads = config["n_heads"]

        assert self.dim_out % self.n_heads == 0, "dim_out should be divisible by n_heads"
        
        self.head_dim = self.dim_out // self.n_heads
        self.W_key = nn.Linear(self.dim_in, self.dim_out, bias=qkv_bias)
        self.W_query = nn.Linear(self.dim_in, self.dim_out, bias=qkv_bias)
        self.W_value = nn.Linear(self.dim_in, self.dim_out, bias=qkv_bias)
        self.dropout = nn.Dropout(config["dropout_rate"])
        self.out_proj = nn.Linear(self.dim_out, self.dim_out)
        self.register_buffer("mask", torch.triu(torch.ones(self.context_length, self.context_length), diagonal=1).bool())

    def forward(self, x):
        batch_size, num_tokens, dim_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        keys = keys.view(batch_size, num_tokens, self.n_heads, self.head_dim)
        queries = queries.view(batch_size, num_tokens, self.n_heads, self.head_dim)
        values = values.view(batch_size, num_tokens, self.n_heads, self.head_dim)

        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        attention_scores = queries @ keys.transpose(2, 3)
        attention_scores.masked_fill_(self.mask[:num_tokens, :num_tokens], -torch.inf)
        attention_weights = torch.softmax(attention_scores / (keys.shape[-1] ** 0.5), dim=-1)
        attention_weights = self.dropout(attention_weights)
        context_vectors = (attention_weights @ values).transpose(1, 2)
        context_vectors = context_vectors.contiguous().view(batch_size, num_tokens, self.dim_out)
        context_vectors = self.out_proj(context_vectors)
        return context_vectors

In [15]:
mma = MaskedMultiheadAttention()

In [16]:
first_batch[0].shape

torch.Size([4, 6])

In [17]:
first_batch[0]

tensor([[  198,  1462,   616,  2583,   408,    25],
        [ 9074,    13, 46606,   536,  5469,   438],
        [   11,  3595,   520,  5493,   438,   292],
        [   13,  1675,   262,  6846,    11,   314]])

In [18]:
preprocessor = InputPreprocessor(tokenizer.n_vocab)

In [19]:
x = preprocessor(first_batch[0])

In [20]:
x.shape

torch.Size([4, 6, 4])

In [21]:
mma(x)

tensor([[[ 1.0817e+00, -1.1796e+00, -3.0451e-01,  7.5470e-01],
         [ 7.9471e-01, -8.3206e-01, -2.4896e-01,  3.7670e-01],
         [ 4.8521e-01, -4.9079e-01, -1.1822e-01,  6.7475e-02],
         [-1.2413e-01, -2.3657e-01,  3.6257e-01, -3.8426e-04],
         [ 2.6919e-01, -6.1060e-01, -7.6509e-03,  7.2203e-02],
         [ 2.3383e-01,  3.2803e-01,  2.9033e-01,  5.0053e-04]],

        [[ 2.9890e-01, -5.0931e-01,  1.0692e-01,  2.9594e-01],
         [-1.9992e-01,  1.0531e-04,  3.1864e-01, -1.7105e-01],
         [-4.1572e-01,  2.3222e-01,  3.9236e-01, -3.4150e-01],
         [-3.8577e-01,  4.0162e-01,  4.9368e-01, -2.8787e-01],
         [ 7.5899e-02, -3.1529e-01,  3.6813e-02, -2.0769e-01],
         [ 1.2757e-01,  3.4122e-01,  2.5406e-01, -9.1902e-02]],

        [[-1.8038e-01, -1.1839e-01,  2.6347e-01,  3.9333e-02],
         [ 6.2026e-03,  1.4262e-01,  1.9263e-01, -2.2400e-01],
         [ 9.5965e-02,  6.8818e-02,  9.3594e-03, -2.4920e-01],
         [-1.8856e-01, -2.7849e-01,  3.0480e-01,  3